# Deep baselines (THANTD, HTD-Net) — paper Table-1 protocol
**Upload-and-run.** Steps: clone the branch → validate both ports on real full-pixel targets (their home regime) → run the paper protocol (Pavia scenario 4, foreign bitumen signature, 5 seeds; additive θ∈{0.075,0.15,0.225} + replacement θ=0.95) → auto-zip + download results & checkpoints.
Training follows each paper exactly (their sample construction and losses); the only substitutions, documented in code, are (a) the single prior signature replaces "a few known target pixels", (b) background selection uses the target-free secondary pixels (our fairness protocol) instead of the detecting image. Runtime ≈ 1–2 h on a T4 GPU.

In [ ]:
!git clone -b colab-deep-baselines --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, torch
sys.path.insert(0, '.')
sys.path.insert(0, 'experiments/spatial')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# OPTIONAL: mount Drive so checkpoints survive disconnects (set MOUNT=True)
MOUNT = False
CKPT_DIR, OUT_DIR = 'ckpt_deep', 'results_deep'
if MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/deep_baselines/ckpt_deep'
    OUT_DIR  = '/content/drive/MyDrive/deep_baselines/results_deep'
import os; os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

## Validation 1 — THANTD on San Diego (real full-pixel targets, paper mode)
Paper mode: CEM coarse detection on the detecting image selects negatives (`bkg_pool=None`). A faithful port should reach high AUC (their papers report ≈0.99 on such scenes; enter the exact paper value below if you have IEEE access).

In [ ]:
import numpy as np, torch
from colab_deep import paper_protocol as P
from thantd_model import THANTD, build_thantd_samples, train_thantd, score_thantd

X, y, sig, known = P.sandiego_validation_sets(n_known=5, seed=0)
rng = np.random.default_rng(0)
a,p,n = build_thantd_samples(X, sig, alpha=0.5, n_samples=1024, rng=rng)  # paper mode (CEM)
m = THANTD(b=X.shape[1]).to(DEVICE) if DEVICE=='cuda' else THANTD(b=X.shape[1])
train_thantd(m, a, p, n, epochs=300, batch_size=64, lr=1e-4, margin=0.3, device=DEVICE)
sc = score_thantd(m, sig, X, device=DEVICE)
from sklearn.metrics import roc_auc_score
print('THANTD San Diego validation AUC =', round(roc_auc_score(y, sc), 4))
PAPER_THANTD_AUC = None  # <- fill from the paper if available

## Validation 2 — HTD-Net on San Diego (paper pipeline end-to-end)
U-AE target generation → LP background selection on the image → SD-CNN (linear/ACE labels) → D(z)=r_t−r_b. Reference points from the HTD-Net paper (their scenes): Moffett 0.9991, WTC 0.9931, HyDICE Forest 0.9949, HyMap 0.9603.

In [ ]:
from colab_deep import htdnet_model as H
uae, scale = H.train_uae(X, epochs=150, device=DEVICE, seed=0, log_every=50)
gen_t = H.generate_targets(uae, scale, sig, X, n_samples=1000, device=DEVICE)
bkg = H.lp_background_selection(X, sig, n_direct=60, n_total=500)
labeler = H.ACELabeler(X)
sd = H.train_sdcnn(gen_t, bkg, labeler, epochs=30, pairs_per_epoch=100_000,
                   device=DEVICE, seed=0, log_every=10)
sc = H.htdnet_detect(sd, X, gen_t, bkg, device=DEVICE)
print('HTD-Net San Diego validation AUC =', round(roc_auc_score(y, sc), 4))

## Main experiment — paper Table-1 protocol (Pavia scenario 4)
Both models are trained per seed from the SAME inputs as our detectors (prior signature + 4026 target-free secondary pixels), each with its own paper-faithful pipeline; the trained model is then scored on all planted cells. Checkpoints save/resume per (model, seed).

In [ ]:
proto = P.build_protocol()

# ---------- THANTD (secondary-data mode: bkg_pool = secondary pixels) ----------
def thantd_fit(tr, sig, seed, ck):
    import os
    m = THANTD(b=tr.shape[1])
    if os.path.exists(ck):
        m.load_state_dict(torch.load(ck, map_location='cpu')); m.to(DEVICE).eval()
        print(f'  resumed {ck}'); return m
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    a,p,n = build_thantd_samples(tr, sig, alpha=0.5, n_samples=1024, rng=rng, bkg_pool=tr)
    m.to(DEVICE)
    train_thantd(m, a, p, n, epochs=300, batch_size=64, lr=1e-4, margin=0.3, device=DEVICE)
    torch.save(m.state_dict(), ck)
    return m

def thantd_scorer(m, proto, seed):
    return lambda planted: score_thantd(m, proto['sig'], planted, device=DEVICE)

res_thantd = P.run_detector('THANTD', thantd_fit, thantd_scorer, proto,
                            out_dir=OUT_DIR, ckpt_dir=CKPT_DIR)

In [ ]:
# ---------- HTD-Net (LP background + ACE labels from the secondary pixels) ----------
def htdnet_fit(tr, sig, seed, ck):
    import os
    if os.path.exists(ck):
        blob = torch.load(ck, map_location='cpu')
        sd = H.SDCNN(); sd.load_state_dict(blob['sdcnn']); sd._scale = blob['scale']
        sd.to(DEVICE).eval()
        print(f'  resumed {ck}')
        return dict(sd=sd, gen_t=blob['gen_t'], bkg=blob['bkg'])
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    uae, sc_ = H.train_uae(tr, epochs=150, device=DEVICE, seed=seed)
    gen_t = H.generate_targets(uae, sc_, sig, tr, n_samples=1000, device=DEVICE, rng=rng)
    bkg = H.lp_background_selection(tr, sig, n_direct=60, n_total=500)
    labeler = H.ACELabeler(tr)
    sd = H.train_sdcnn(gen_t, bkg, labeler, epochs=30, pairs_per_epoch=100_000,
                       device=DEVICE, seed=seed, log_every=10)
    torch.save(dict(sdcnn=sd.state_dict(), scale=sd._scale, gen_t=gen_t, bkg=bkg), ck)
    return dict(sd=sd, gen_t=gen_t, bkg=bkg)

def htdnet_scorer(state, proto, seed):
    rng = np.random.default_rng(seed)
    return lambda planted: H.htdnet_detect(state['sd'], planted, state['gen_t'],
                                           state['bkg'], device=DEVICE, rng=rng)

res_htdnet = P.run_detector('HTDNet', htdnet_fit, htdnet_scorer, proto,
                            out_dir=OUT_DIR, ckpt_dir=CKPT_DIR)

In [ ]:
# ---------- Summary + auto-download ----------
import numpy as np
for name, res in [('THANTD', res_thantd), ('HTD-Net', res_htdnet)]:
    print(f'\n### {name} (AUC mean+/-std, 5 seeds)')
    for k, v in res.items():
        print(f'  {k}: {np.mean(v):.3f}+/-{np.std(v):.3f}')
P.zip_and_download(dirs=(OUT_DIR, CKPT_DIR))